In [1]:
# -*- coding: utf-8 -*-
import sys
from pathlib import Path
import pandas as pd
from typing import Final, List

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.utils import clean_text, chunk_text, _META_COLS

/home/jd/Documentos/CODIGO-2025/Machine-Learning-2025/src/ML/tutorials/rag-openai-chats/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def preprocess(
    jsonl_path: Path | str,
    out_parquet_chunks: Path | str,
    out_parquet_raw: Path | str | None = None,
) -> None:
    """Transforma el *messages.jsonl* exportado por *parser.py* en dos Parquet:

    1. **df_chunks** : un row por chunk + metadatos necesarios para indexar.
    2. **df_raw  ** : (opcional) dump 1‑a‑1 de los mensajes completos.
    """

    df = pd.read_json(
        jsonl_path,
        lines=True,
        dtype_backend="pyarrow",  # mantiene UUIDs como strings
    )

    # Limpiar texto base
    df["clean"] = df["text"].map(clean_text, na_action="ignore")

    # Chunking
    df["chunks"] = df["clean"].map(chunk_text, na_action="ignore")

    # Explode y selección de columnas
    df_chunks = (
        df[_META_COLS + ["chunks"]]
        .explode("chunks")
        .dropna(subset=["chunks"]) 
        .rename(columns={"chunks": "chunk_text"})
        .reset_index(drop=True)
    )

    # Índice de chunk dentro de Mensaje
    df_chunks["chunk_index"] = df_chunks.groupby("message_id").cumcount()

    # Guardar
    out_parquet_chunks = Path(out_parquet_chunks)
    df_chunks.to_parquet(out_parquet_chunks, index=False)
    if out_parquet_raw is not None:
        out_parquet_raw = Path(out_parquet_raw)
        df.to_parquet(out_parquet_raw, index=False)

    # Logs
    print(f"OK: {len(df):,}")
    print(f"OK: {len(df_chunks):,}")
    return df, df_chunks

In [3]:
jsonl_path          = "/home/jd/Documentos/CODIGO-2025/Machine-Learning-2025/src/ML/tutorials/rag-openai-chats/data/interim/messages.jsonl"
parquet_chunks_path = "/home/jd/Documentos/CODIGO-2025/Machine-Learning-2025/src/ML/tutorials/rag-openai-chats/data/processed/chunks.parquet"
parquet_raw_path    = "/home/jd/Documentos/CODIGO-2025/Machine-Learning-2025/src/ML/tutorials/rag-openai-chats/data/processed/messages_raw.parquet"

df_raw, df_chunks = preprocess(
    Path(jsonl_path),
    out_parquet_chunks=Path(parquet_chunks_path),
    out_parquet_raw=Path(parquet_raw_path)
)

OK: 31,193
OK: 224,807


In [4]:
df_chunks

,message_id,parent_id,conversation_id,depth,order_in_conv,role,model_slug,conversation_ttl,created_at,updated_at,chunk_text,chunk_index
0,bbb21b51-9889-499c-8805-3975f3ec2520,33007a2c-1523-4efe-a42d-5219fd469df4,0930598c-baaf-55c9-8117-af1d96d152e4,3,3,user,gpt-4o,ICDC Funciones y Evaluación,2025-04-03 05:30:47.508710,<NA>,Qué es el IDCD PROPUESTO Y CUÁLES SON SUS FUNC...,0
1,13513ae1-ec28-46d0-81b6-b179516aeffb,ef64065d-2dfd-41a4-bccd-17816af24a76,0930598c-baaf-55c9-8117-af1d96d152e4,5,5,tool,gpt-4o,ICDC Funciones y Evaluación,2025-04-03 05:30:59.584338,<NA>,\n## Documento SOTA: Estado del Arte en Interp...,0
2,13513ae1-ec28-46d0-81b6-b179516aeffb,ef64065d-2dfd-41a4-bccd-17816af24a76,0930598c-baaf-55c9-8117-af1d96d152e4,5,5,tool,gpt-4o,ICDC Funciones y Evaluación,2025-04-03 05:30:59.584338,<NA>,"nte, descomponiendo su computación en unidades...",1
3,13513ae1-ec28-46d0-81b6-b179516aeffb,ef64065d-2dfd-41a4-bccd-17816af24a76,0930598c-baaf-55c9-8117-af1d96d152e4,5,5,tool,gpt-4o,ICDC Funciones y Evaluación,2025-04-03 05:30:59.584338,<NA>,r respuestas. (Ref: Todos los papers)\n* **S...,2
4,13513ae1-ec28-46d0-81b6-b179516aeffb,ef64065d-2dfd-41a4-bccd-17816af24a76,0930598c-baaf-55c9-8117-af1d96d152e4,5,5,tool,gpt-4o,ICDC Funciones y Evaluación,2025-04-03 05:30:59.584338,<NA>,dades de Análisis Fundamentales:**\n\n* **Sp...,3
...,...,...,...,...,...,...,...,...,...,...,...,...
224802,3b8832ad-7f63-4f33-8ebf-801b8c8640b5,aaa2372b-e3ef-4349-a777-dec364b35a56,0c04f394-8449-5d80-ae2c-6646b83d7222,9,9,assistant,auto,Cadenas en LangChain,2024-10-04 17:34:35.567951,<NA>,"o, necesitamos pasarla sin cambios para ser us...",6
224803,3b8832ad-7f63-4f33-8ebf-801b8c8640b5,aaa2372b-e3ef-4349-a777-dec364b35a56,0c04f394-8449-5d80-ae2c-6646b83d7222,9,9,assistant,auto,Cadenas en LangChain,2024-10-04 17:34:35.567951,<NA>,text\n Answer the question based only on the ...,7
224804,3b8832ad-7f63-4f33-8ebf-801b8c8640b5,aaa2372b-e3ef-4349-a777-dec364b35a56,0c04f394-8449-5d80-ae2c-6646b83d7222,9,9,assistant,auto,Cadenas en LangChain,2024-10-04 17:34:35.567951,<NA>,omplejo) y lo convierte en un formato de texto...,8
224805,3b8832ad-7f63-4f33-8ebf-801b8c8640b5,aaa2372b-e3ef-4349-a777-dec364b35a56,0c04f394-8449-5d80-ae2c-6646b83d7222,9,9,assistant,auto,Cadenas en LangChain,2024-10-04 17:34:35.567951,<NA>,"exto relevante, en este caso, ""harrison worked...",9
